# Day 13 — Complete Uncertainty and Business Validation

**Project:** EconoCausal — Dynamic Pricing via Double Machine Learning  
**Stage:** Uncertainty Propagation, Sensitivity Analysis, and Business Constraint Validation  

---

## Executive Summary & Objectives

In Days 10–12, we validated the Double Machine Learning (DML) treatment effect of price on sales demand, built a local linear what-if pricing simulator, and generated an inventory-constrained pricing recommendation prototype.

The primary objective of **Day 13** is to rigorously audit and stress-test the pricing recommendations against:
1. **Statistical Uncertainty:** How do candidate prices, demand forecasts, and revenue estimates behave under the lower and upper bounds of the 95% confidence intervals from both DML specifications (Spec 1 core controls vs. Spec 2 with inventory)?
2. **Recommendation Stability:** Does the top-ranked pricing scenario (+10% price increase) remain stable across coefficient bounds, or is that stability an artifact of an imprecise, near-zero point estimate?
3. **Threshold / Breakeven Analysis:** What critical price sensitivity $(\theta^*)$ or elasticity $(\epsilon^*)$ would be required to reverse the recommendation?
4. **Business Constraints & Integrity Rules:** Validate that candidate prices are positive and within observed historical support, demand is non-negative, revenue matches $P \times \min(\hat{Y}, \text{Inventory})$, inventory constraints are verified, and costs/margins are explicitly audited.
5. **Causal vs. Observational Validity:** Document the critical distinctions between Double Machine Learning under unconfoundedness vs. causal identification in real retail environments with unobserved promotions and competitor reactions.


In [1]:
from IPython.display import display
# Cell 1: Environment check and imports
import os
import sys
import json
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Set clean visualization style
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Helvetica", "Arial"]
plt.rcParams["axes.edgecolor"] = "#cccccc"
plt.rcParams["axes.linewidth"] = 0.8

# Resolve project root dynamically and independently
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in locals() else CURRENT_DIR

# Add project root to sys.path to enable importing reusable modules from src/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.business_validation import (
    load_and_validate_inputs,
    run_uncertainty_sensitivity,
    validate_business_rules
)

print(f"Project root resolved to: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")


Project root resolved to: /Users/hasikaippili/Desktop/EconoCausal
Python executable: /Users/hasikaippili/Desktop/EconoCausal/.venv/bin/python


## 1 · Data and Model Inputs Validation

We load the verified outputs from previous project milestones:
- `data/processed/dml_validated_results.json` (from Day 10 DML validation)
- `data/processed/pricing_simulator_results.json` (from Day 11 pricing simulator)
- `data/processed/dml_preparation.csv` (prepared modeling dataset)

We enforce strict validation on data types, non-null constraints, positive baseline prices, non-negative demand, and expected schema keys.


In [2]:
# Cell 2: Load and validate inputs
DML_RESULTS_PATH = PROJECT_ROOT / "data" / "processed" / "dml_validated_results.json"
SIM_RESULTS_PATH = PROJECT_ROOT / "data" / "processed" / "pricing_simulator_results.json"
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dml_preparation.csv"

# Load using the robust validation function from src/business_validation
dml_res, sim_res, df_raw = load_and_validate_inputs(
    dml_path=DML_RESULTS_PATH,
    sim_path=SIM_RESULTS_PATH,
    data_path=DATA_PATH
)

print("SUCCESS: All input files loaded and schema validated.")
print("-" * 75)
print(f"Baseline Product: {sim_res['product_id']} | Store: {sim_res['store_id']}")
print(f"Baseline Price: ${sim_res['baseline_price']:.2f}")
print(f"Baseline Units Sold: {sim_res['baseline_units']:.1f}")
print(f"Baseline Revenue: ${sim_res['baseline_revenue']:.2f}")
print(f"Baseline Inventory Level: {sim_res['baseline_inventory']:.0f} units (provisional constraint)")
print(f"Observed Historical Price Range: ${sim_res['price_obs_min']:.2f} to ${sim_res['price_obs_max']:.2f}")


SUCCESS: All input files loaded and schema validated.
---------------------------------------------------------------------------
Baseline Product: P0001 | Store: S001
Baseline Price: $55.13
Baseline Units Sold: 173.0
Baseline Revenue: $9537.49
Baseline Inventory Level: 195 units (provisional constraint)
Observed Historical Price Range: $10.26 to $99.97


## 2 · Estimator Scale, Confidence Intervals, and Uncertainty Bounds

### Estimator Formulation and Scale
In Days 9–10, the Double Machine Learning model was trained with:
- **Treatment ($T$):** Raw unit price (`Price` in dollars).
- **Outcome ($Y$):** Raw units sold (`Units Sold`).
- **Nuisance models:** 3-fold cross-fitting using `ExtraTreesRegressor(n_estimators=100)`.
- **Debiased parameter $(\theta)$:** Estimated via OLS of residual outcome on residual price without intercept.

Because both $T$ and $Y$ are entered in their raw, unscaled natural units (dollars and units), the coefficient $\theta$ represents:
$$\Delta \text{Units Sold} = \theta \times \Delta \text{Price}$$
$$\theta \equiv \frac{\partial Y}{\partial P} \quad [\text{units sold per } \$1.00 \text{ price change}]$$

### Statistical Significance and Robust Standard Errors
The standard errors and 95% confidence intervals were computed using **HC3 heteroskedasticity-consistent robust standard errors** on the pooled residual regression.
Let us examine the parameters across specifications:


In [3]:
# Cell 3: Display DML specifications and statistical metrics
spec_summary = pd.DataFrame([
    {
        "Specification": "Spec 1 (Core Controls)",
        "Controls": "Product, Store, Category, Region, Year, Month, DayOfWeek",
        "theta (DML Coeff)": dml_res["spec1"]["theta"],
        "Std Err (HC3)": dml_res["spec1"]["std_err"],
        "95% CI Lower": dml_res["spec1"]["ci_low"],
        "95% CI Upper": dml_res["spec1"]["ci_high"],
        "p-value": dml_res["spec1"]["p_value"],
        "Statistically Sig (alpha=0.05)": dml_res["spec1"]["p_value"] < 0.05,
    },
    {
        "Specification": "Spec 2 (+ Inventory Level)",
        "Controls": "Spec 1 + Inventory Level",
        "theta (DML Coeff)": dml_res["spec2"]["theta"],
        "Std Err (HC3)": dml_res["spec2"]["std_err"],
        "95% CI Lower": dml_res["spec2"]["ci_low"],
        "95% CI Upper": dml_res["spec2"]["ci_high"],
        "p-value": dml_res["spec2"]["p_value"],
        "Statistically Sig (alpha=0.05)": dml_res["spec2"]["p_value"] < 0.05,
    }
])

display(spec_summary.style.format({
    "theta (DML Coeff)": "{:+.6f}",
    "Std Err (HC3)": "{:.6f}",
    "95% CI Lower": "{:+.6f}",
    "95% CI Upper": "{:+.6f}",
    "p-value": "{:.4f}",
}))

# Calculate implied price elasticity of demand: epsilon = theta * (P_base / Y_base)
base_p = sim_res["baseline_price"]
base_y = sim_res["baseline_units"]
elasticity_spec1 = dml_res["spec1"]["theta"] * (base_p / base_y)
elasticity_spec2 = dml_res["spec2"]["theta"] * (base_p / base_y)

print(f"Implied Price Elasticity of Demand at Baseline ($55.13, 173 units):")
print(f"  Spec 1: epsilon = {elasticity_spec1:+.4f} (virtually zero, completely inelastic)")
print(f"  Spec 2: epsilon = {elasticity_spec2:+.4f} (near-zero, completely inelastic)")


Implied Price Elasticity of Demand at Baseline ($55.13, 173 units):
  Spec 1: epsilon = +0.0034 (virtually zero, completely inelastic)
  Spec 2: epsilon = -0.0062 (near-zero, completely inelastic)


## 3 · Sensitivity Analysis Across Coefficient Bounds and Critical Thresholds

> [!IMPORTANT]
> **Methodological Clarification:**  
> Propagating the lower and upper bounds of the DML coefficient $\theta$ into the demand equation is a **parameter sensitivity analysis**, *not* a formal confidence interval for predicted revenue or optimal price. A formal revenue prediction interval requires full joint covariance modeling of baseline demand, nuisance estimation errors, and residual variance.

### Scenarios Evaluated
We evaluate candidate prices across the five Day 11 pricing percentage changes:
$$-10\%, \quad -5\%, \quad 0\% \text{ (baseline)}, \quad +5\%, \quad +10\%$$

Across the following seven coefficient scenarios:
1. **Spec 1 Point Estimate:** $\theta = +0.010699$ (Core primary DML estimate)
2. **Spec 1 CI Lower 95%:** $\theta = -0.019671$ (Lower bound of Spec 1 HC3 interval)
3. **Spec 1 CI Upper 95%:** $\theta = +0.041069$ (Upper bound of Spec 1 HC3 interval)
4. **Spec 2 Point Estimate:** $\theta = -0.019468$ (Alternative specification adding inventory)
5. **Spec 2 CI Lower 95%:** $\theta = -0.043963$ (Lower bound of Spec 2 95% interval)
6. **Spec 2 CI Upper 95%:** $\theta = +0.005027$ (Upper bound of Spec 2 95% interval)
7. **Analytical Breakeven Threshold:** The critical price sensitivity $\theta^*$ where raising price by $+10\%$ produces zero incremental revenue over the baseline:
   $$R(P_{+10\%}) \le R(P_{\text{base}}) \iff \theta^* \le \frac{\frac{R_{\text{base}}}{P_{+10\%}} - Y_{\text{base}}}{P_{+10\%} - P_{\text{base}}} \approx -2.8528 \text{ units/\$}$$


In [4]:
# Cell 4: Execute multi-scenario sensitivity analysis
sensitivity_df = run_uncertainty_sensitivity(
    base_price=sim_res["baseline_price"],
    base_units=sim_res["baseline_units"],
    base_revenue=sim_res["baseline_revenue"],
    base_inventory=sim_res["baseline_inventory"],
    dml_results=dml_res,
    percentage_changes=[-0.10, -0.05, 0.00, 0.05, 0.10],
    observed_price_min=sim_res["price_obs_min"],
    observed_price_max=sim_res["price_obs_max"],
    include_threshold_analysis=True
)

print(f"Generated {len(sensitivity_df)} sensitivity scenario records across 7 parameter settings.")


Generated 35 sensitivity scenario records across 7 parameter settings.


In [5]:
# Cell 5: Detailed Sensitivity Results Table
display_cols = [
    "specification",
    "coefficient_scenario",
    "theta",
    "pricing_scenario",
    "candidate_price",
    "estimated_demand",
    "inventory_constrained_demand",
    "estimated_revenue",
    "revenue_change_pct",
    "rank_in_scenario"
]

sensitivity_df["full_scenario"] = (
    sensitivity_df["specification"] + " [" + sensitivity_df["coefficient_scenario"] + "]"
)

# Primary Spec 1 + Breakeven Threshold pivot
spec1_mask = sensitivity_df["specification"].isin(["Spec 1 (Core)", "Stress-Test (Elastic)"])
pivot_rev_spec1 = sensitivity_df[spec1_mask].pivot(
    index="pricing_scenario",
    columns="coefficient_scenario",
    values="estimated_revenue"
)[["Point Estimate", "CI Lower 95%", "CI Upper 95%", "Breakeven Threshold"]]

pivot_demand_spec1 = sensitivity_df[spec1_mask].pivot(
    index="pricing_scenario",
    columns="coefficient_scenario",
    values="estimated_demand"
)[["Point Estimate", "CI Lower 95%", "CI Upper 95%", "Breakeven Threshold"]]

# Spec 2 (+Inventory) sensitivity pivot
spec2_mask = sensitivity_df["specification"] == "Spec 2 (+Inventory)"
pivot_rev_spec2 = sensitivity_df[spec2_mask].pivot(
    index="pricing_scenario",
    columns="coefficient_scenario",
    values="estimated_revenue"
)[["Point Estimate", "CI Lower 95%", "CI Upper 95%"]]

print("SPECIFICATION 1 (CORE) ESTIMATED REVENUE ($):")
display(pivot_rev_spec1.style.format("${:,.2f}"))

print("\nSPECIFICATION 1 (CORE) ESTIMATED DEMAND (UNITS):")
display(pivot_demand_spec1.style.format("{:.2f}"))

print("\nSPECIFICATION 2 (+INVENTORY SENSITIVITY) ESTIMATED REVENUE ($):")
display(pivot_rev_spec2.style.format("${:,.2f}"))


SPECIFICATION 1 (CORE) ESTIMATED REVENUE ($):

SPECIFICATION 1 (CORE) ESTIMATED DEMAND (UNITS):

SPECIFICATION 2 (+INVENTORY SENSITIVITY) ESTIMATED REVENUE ($):


## 4 · Visualizing Uncertainty and Pricing Sensitivity

We plot:
1. **Demand vs. Candidate Price:** Illustrating how estimated demand responds under the DML point estimate, confidence interval bounds, and the critical breakeven threshold.
2. **Revenue vs. Candidate Price:** Comparing estimated revenue trajectories.
3. **Revenue Change (%) relative to Baseline:** Revealing why the $+10\%$ scenario dominates across all DML empirical bounds.


In [6]:
# Cell 6: Visualizations of Sensitivity and Uncertainty
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

color_map = {
    "Point Estimate": "#1f77b4",       # Solid blue
    "CI Lower 95%": "#ff7f0e",         # Orange
    "CI Upper 95%": "#2ca02c",         # Green
    "Breakeven Threshold": "#d62728",  # Dashed red
}

# 1. Demand vs Price (Spec 1 + Threshold)
ax1 = axes[0]
spec1_data = sensitivity_df[
    (sensitivity_df["specification"] == "Spec 1 (Core)") |
    (sensitivity_df["specification"] == "Stress-Test (Elastic)")
]

for scenario_name, group in spec1_data.groupby("coefficient_scenario"):
    linestyle = "--" if "Threshold" in scenario_name else "-"
    marker = "s" if "Threshold" in scenario_name else "o"
    ax1.plot(
        group["candidate_price"],
        group["estimated_demand"],
        label=f"{scenario_name} (θ={group['theta'].iloc[0]:+.4f})",
        color=color_map.get(scenario_name, "#333333"),
        linestyle=linestyle,
        marker=marker,
        linewidth=2.2,
        markersize=6
    )

ax1.axhline(sim_res["baseline_inventory"], color="#7f7f7f", linestyle=":", label=f"Inv Cap ({sim_res['baseline_inventory']:.0f})")
ax1.set_title("Estimated Demand vs Candidate Price", fontsize=13, fontweight="bold")
ax1.set_xlabel("Candidate Price ($)", fontsize=11)
ax1.set_ylabel("Estimated Units Sold", fontsize=11)
ax1.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f"))
ax1.xaxis.set_major_formatter(ticker.FormatStrFormatter("$%.2f"))
ax1.legend(loc="best", fontsize=9)

# 2. Revenue vs Price
ax2 = axes[1]
for scenario_name, group in spec1_data.groupby("coefficient_scenario"):
    linestyle = "--" if "Threshold" in scenario_name else "-"
    marker = "s" if "Threshold" in scenario_name else "o"
    ax2.plot(
        group["candidate_price"],
        group["estimated_revenue"],
        label=f"{scenario_name}",
        color=color_map.get(scenario_name, "#333333"),
        linestyle=linestyle,
        marker=marker,
        linewidth=2.2,
        markersize=6
    )

ax2.axhline(sim_res["baseline_revenue"], color="#7f7f7f", linestyle=":", label=f"Baseline Rev (${sim_res['baseline_revenue']:,.0f})")
ax2.set_title("Estimated Revenue vs Candidate Price", fontsize=13, fontweight="bold")
ax2.set_xlabel("Candidate Price ($)", fontsize=11)
ax2.set_ylabel("Estimated Revenue ($)", fontsize=11)
ax2.yaxis.set_major_formatter(ticker.StrMethodFormatter("${x:,.0f}"))
ax2.xaxis.set_major_formatter(ticker.FormatStrFormatter("$%.2f"))
ax2.legend(loc="best", fontsize=9)

# 3. Revenue Change (%) by Scenario
ax3 = axes[2]
pivot_pct = spec1_data.pivot(
    index="pricing_scenario",
    columns="coefficient_scenario",
    values="revenue_change_pct"
)[["Point Estimate", "CI Lower 95%", "CI Upper 95%", "Breakeven Threshold"]]

x = np.arange(len(pivot_pct))
width = 0.20
for i, col in enumerate(pivot_pct.columns):
    ax3.bar(
        x + i * width,
        pivot_pct[col],
        width,
        label=col,
        color=color_map.get(col, "#333333"),
        alpha=0.85
    )

ax3.axhline(0, color="black", linewidth=0.8)
ax3.set_xticks(x + width * 1.5)
ax3.set_xticklabels(pivot_pct.index)
ax3.set_title("Revenue Change vs Baseline (%)", fontsize=13, fontweight="bold")
ax3.set_xlabel("Pricing Scenario", fontsize=11)
ax3.set_ylabel("Revenue Change (%)", fontsize=11)
ax3.yaxis.set_major_formatter(ticker.PercentFormatter(decimals=1))
ax3.legend(loc="best", fontsize=8.5)

plt.tight_layout()

# Save chart to reports/
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CHART_PATH = REPORTS_DIR / "day13_sensitivity_analysis.png"
fig.savefig(CHART_PATH, dpi=300, bbox_inches="tight")
print(f"Sensitivity visualization successfully saved to: {CHART_PATH}")
plt.show()


Sensitivity visualization successfully saved to: /Users/hasikaippili/Desktop/EconoCausal/reports/day13_sensitivity_analysis.png


## 5 · Business Constraints and Commercial Rules Audit

We rigorously audit five critical operational dimensions:
1. **Price Limits and Support Validation:** Do candidate prices comply with positive price bounds and fall strictly within historical observations without extrapolation?
2. **Demand Non-Negativity:** Is demand strictly non-negative across all candidate scenarios?
3. **Revenue Calculation Integrity:** Does revenue strictly equal $P \times \min(\hat{Y}, \text{Inventory})$?
4. **Inventory Constraints & Timing Ambiguity:** Does available inventory constrain sales at the time of purchase?
5. **Profit vs. Revenue Separation:** Does the dataset support profit optimization, or must revenue maximization be strictly isolated?


In [7]:
# Cell 7: Execute Business Constraints Audit
audit_results = validate_business_rules(
    sensitivity_df=sensitivity_df,
    product_df=df_raw,
    dataset_has_costs=False
)

print("=" * 80)
print("BUSINESS AND DATA INTEGRITY AUDIT RESULTS:")
print("=" * 80)
print(f"1. Candidate Prices Positive: {audit_results['all_prices_positive']} "
      f"(Range: ${audit_results['min_candidate_price']:.2f} to ${audit_results['max_candidate_price']:.2f})")
print(f"2. Extrapolation Check: No prices outside historical support? {not audit_results['any_extrapolation']} "
      f"(Historical bounds: ${sim_res['price_obs_min']:.2f} - ${sim_res['price_obs_max']:.2f})")
print(f"3. Demand Non-Negativity: {audit_results['all_demand_non_negative']} "
      f"(Minimum estimated demand: {audit_results['min_estimated_demand']:.2f} units)")
print(f"4. Revenue Calculation Verified: {audit_results['revenue_calculation_correct']} "
      f"(P * inventory_constrained_demand verified across all rows)")
print(f"5. Inventory Cap Binding: {audit_results['inventory_binding_count']} scenarios binding "
      f"(Estimated demand ~173 units < 195 provisional cap)")
print(f"6. Recommendation Stable Across DML Bounds: {audit_results['is_stable_across_dml_bounds']} "
      f"(Top candidate across all DML bounds: {audit_results['stat_scenarios_top_recommendation']})")
print(f"7. Unit Costs Available: {audit_results['dataset_has_costs']} "
      f"(Profit optimization possible: {audit_results['profit_optimization_possible']})")
print("-" * 80)
print(f"PROFIT AUDIT WARNING:\n{audit_results['profit_warning']}")
print("-" * 80)
print(f"CAUSAL VALIDITY AUDIT WARNING:\n{audit_results['causal_warning']}")
print("=" * 80)


BUSINESS AND DATA INTEGRITY AUDIT RESULTS:
1. Candidate Prices Positive: True (Range: $49.62 to $60.64)
2. Extrapolation Check: No prices outside historical support? True (Historical bounds: $10.26 - $99.97)
3. Demand Non-Negativity: True (Minimum estimated demand: 157.28 units)
4. Revenue Calculation Verified: True (P * inventory_constrained_demand verified across all rows)
5. Inventory Cap Binding: 0 scenarios binding (Estimated demand ~173 units < 195 provisional cap)
6. Recommendation Stable Across DML Bounds: True (Top candidate across all DML bounds: ['+10%'])
7. Unit Costs Available: False (Profit optimization possible: False)
--------------------------------------------------------------------------------
PROFIT AUDIT WARNING:
CRITICAL: Dataset contains no cost of goods sold (COGS), carrying costs, or margins. Estimated revenue CANNOT be interpreted as profit. A higher price may reduce sales volume and total revenue while increasing or decreasing profit depending on unit margin

## 6 · Counterfactual Inventory Stress-Test

In the baseline observation, the inventory cap was **195 units**, while demand was estimated at **~173 units**. Thus, the inventory cap did not bind.

To test how the decision logic responds when inventory *does* constrain demand, we simulate a counterfactual stockout scenario where available inventory is restricted to **160 units** (below the baseline demand of 173 units).


In [8]:
# Cell 8: Counterfactual Binding Inventory Simulation (Cap = 160 units)
COUNTERFACTUAL_INV_CAP = 160.0

cf_rows = []
base_p = sim_res["baseline_price"]
base_y = sim_res["baseline_units"]
theta_spec1 = dml_res["spec1"]["theta"]

for pct in [-0.10, -0.05, 0.00, 0.05, 0.10]:
    cand_p = round(base_p * (1.0 + pct), 2)
    delta_p = cand_p - base_p
    unconstrained_demand = max(0.0, base_y + theta_spec1 * delta_p)
    constrained_demand = min(unconstrained_demand, COUNTERFACTUAL_INV_CAP)
    is_binding = unconstrained_demand > COUNTERFACTUAL_INV_CAP
    rev = round(cand_p * constrained_demand, 2)
    
    cf_rows.append({
        "Pricing Scenario": f"{pct:+.0%}",
        "Candidate Price": cand_p,
        "Unconstrained Demand": round(unconstrained_demand, 2),
        "Inventory Cap": COUNTERFACTUAL_INV_CAP,
        "Constrained Demand": round(constrained_demand, 2),
        "Inventory Binds?": is_binding,
        "Constrained Revenue ($)": rev,
        "Lost Sales (Stockout Units)": round(unconstrained_demand - constrained_demand, 2)
    })

cf_df = pd.DataFrame(cf_rows)
print("COUNTERFACTUAL INVENTORY RESTRICTION (Cap = 160 Units):")
display(cf_df.set_index("Pricing Scenario"))
print("\nFINDING: When inventory strictly binds, sales volume is capped at 160 units across all scenarios.")
print("In this case, raising price to +10% ($60.64) yields higher revenue ($9,702.40) without any volume penalty,")
print("because excess unfulfillable demand is eliminated without sacrificing actual sales.")


COUNTERFACTUAL INVENTORY RESTRICTION (Cap = 160 Units):
                  Candidate Price  ...  Lost Sales (Stockout Units)
Pricing Scenario                   ...                             
-10%                        49.62  ...                        12.94
-5%                         52.37  ...                        12.97
+0%                         55.13  ...                        13.00
+5%                         57.89  ...                        13.03
+10%                        60.64  ...                        13.06

[5 rows x 7 columns]

FINDING: When inventory strictly binds, sales volume is capped at 160 units across all scenarios.
In this case, raising price to +10% ($60.64) yields higher revenue ($9,702.40) without any volume penalty,
because excess unfulfillable demand is eliminated without sacrificing actual sales.


## 7 · Export Deliverables

We save the full validation dataset to:
`reports/day13_sensitivity_results.csv`


In [9]:
# Cell 9: Export Validation Artifacts
EXPORT_PATH = REPORTS_DIR / "day13_sensitivity_results.csv"

# Reorder columns cleanly for export
export_cols = [
    "specification",
    "coefficient_scenario",
    "theta",
    "pricing_scenario",
    "candidate_price",
    "baseline_units",
    "estimated_demand",
    "inventory_cap",
    "inventory_constrained_demand",
    "inventory_binding",
    "estimated_revenue",
    "revenue_change_dollars",
    "revenue_change_pct",
    "rank_in_scenario",
    "extrapolates"
]

sensitivity_df[export_cols].to_csv(EXPORT_PATH, index=False)
print(f"SUCCESS: Exported {len(sensitivity_df)} validation rows to: {EXPORT_PATH}")

# Verify file size and existence
print(f"File size on disk: {EXPORT_PATH.stat().st_size:,} bytes")


SUCCESS: Exported 35 validation rows to: /Users/hasikaippili/Desktop/EconoCausal/reports/day13_sensitivity_results.csv
File size on disk: 4,184 bytes


## 8 · Decision-Validation Summary, Limitations, and Commercial Readiness

### Key Findings & Answers to Core Evaluation Questions

#### 1. Does the recommendation change across sensitivity scenarios?
- Across all empirical DML bounds (Spec 1 point estimate, CI low, CI high, Spec 2 point estimate, CI low, CI high), the **+10% price scenario ($60.64) consistently ranks #1**.
- **Crucial Nuance:** This apparent "stability" is **spurious**. Because the estimated DML coefficient is statistically indistinguishable from zero ($\theta = +0.0107, p = 0.49$) and tightly bounded ($|\theta| \le 0.044$), demand is modeled as virtually completely inelastic ($|\epsilon| < 0.014$). In any model where demand does not decline with price, revenue mechanically maximizes at the highest evaluated price.
- Only when price sensitivity reaches the **critical threshold of $\theta^* \le -2.8528$** (elasticity $|\epsilon| > 0.91$) does the $+10\%$ recommendation fail to improve revenue over baseline.

#### 2. Is the estimated price effect sufficiently precise for commercial pricing decisions?
- **No.** The confidence interval spans zero in both specifications (Spec 1: $[-0.0197, +0.0411]$, $p = 0.4899$). We cannot reject the null hypothesis that price has zero effect on demand, nor can we rule out moderate negative or positive associations.
- The coefficient shifts by **282%** (from $+0.0107$ to $-0.0195$) simply by including `Inventory Level`, indicating acute sensitivity to model specification.

#### 3. Does the available data support the chosen inventory constraints?
- **No.** The column `Inventory Level` has unknown timing. In retail forecasting, if inventory represents end-of-period closing stock, it is a downstream consequence of sales (a "bad control"), introducing collider bias. If it represents opening stock, it is a valid pre-treatment constraint. Without timestamped transaction logs, any inventory cap is strictly provisional.

#### 4. Limitations and Causal Validity
- **Omitted Confounders:** The dataset contains no records of promotional discounts, markdowns, competitor shelf prices, seasonal holidays, marketing spend, or store footfall.
- **Observational vs. Causal:** Double Machine Learning debiases high-dimensional controls under the conditional unconfoundedness assumption. Because key determinants of retail pricing and demand are unobserved, unconfoundedness does not hold. The estimate represents an observational conditional correlation, not a structural causal price elasticity.
- **Revenue vs. Profit:** Unit costs, holding costs, and margins are missing. Setting prices to maximize top-line revenue can easily destroy gross margin if marginal costs are positive.

#### 5. Required Steps Before Commercial Deployment
1. **Randomized Pricing Experiments (A/B Testing):** Run randomized price rollouts across matched store clusters to identify true causal elasticity free from unobserved confounding.
2. **Cost of Goods Sold (COGS) Ingestion:** Integrate unit costs to optimize gross margin: $\Pi(P) = (P - c) \cdot Q(P)$.
3. **High-Frequency Point-of-Sale (POS) Logs:** Capture promotional flags, competitor prices, and opening inventory.


In [10]:
# Cell 10: Final Executive Decision-Validation Report
print("=" * 80)
print("ECONOCAUSAL DAY 13: FINAL DECISION-VALIDATION REPORT")
print("=" * 80)
print(f"Artifacts Created:")
print(f"  - Validation Table : {EXPORT_PATH.relative_to(PROJECT_ROOT)}")
print(f"  - Sensitivity Chart: {CHART_PATH.relative_to(PROJECT_ROOT)}")
print("-" * 80)
print("Key Decision Findings:")
print(f"  - Top Ranked Candidate across all DML Bounds: +10% (${sim_res['baseline_price']*1.10:.2f})")
print(f"  - Implied DML Demand Elasticity: {elasticity_spec1:+.4f} (virtually zero)")
print(f"  - Critical Inversion Threshold (Breakeven): theta = {dml_res['spec1']['theta']:.4f} vs threshold = {((sim_res['baseline_revenue']/(sim_res['baseline_price']*1.10)) - sim_res['baseline_units'])/(sim_res['baseline_price']*0.10):.4f}")
print(f"  - Commercial Readiness Status: NOT READY FOR AUTONOMOUS DEPLOYMENT (Research Prototype Only)")
print("=" * 80)


ECONOCAUSAL DAY 13: FINAL DECISION-VALIDATION REPORT
Artifacts Created:
  - Validation Table : reports/day13_sensitivity_results.csv
  - Sensitivity Chart: reports/day13_sensitivity_analysis.png
--------------------------------------------------------------------------------
Key Decision Findings:
  - Top Ranked Candidate across all DML Bounds: +10% ($60.64)
  - Implied DML Demand Elasticity: +0.0034 (virtually zero)
  - Critical Inversion Threshold (Breakeven): theta = 0.0107 vs threshold = -2.8528
  - Commercial Readiness Status: NOT READY FOR AUTONOMOUS DEPLOYMENT (Research Prototype Only)
